# GPT 시리즈: 언어모델의 스케일링 - 실습 코드 3: GPT 스타일 언어모델 학습 루프 (PyTorch)

- Tutorial ID: `expand-gpt-series`
- Tutorial: GPT 시리즈: 언어모델의 스케일링
- Section ID: `expand-gpt-series-code-3`
- Section: 실습 코드 3: GPT 스타일 언어모델 학습 루프 (PyTorch)


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 3: GPT 스타일 언어모델 학습 루프 (PyTorch)
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#   3) 미래 토큰을 -inf로 막은 뒤 softmax 확률이 0이 되는지 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleGPT(nn.Module):
    """GPT-2 스타일 언어모델 (간소화 버전)"""
        def __init__(self, vocab_size, d_model=768, n_heads=12, n_layers=12):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(1024, d_model)
        self.blocks = nn.ModuleList([
            nn.TransformerDecoderLayer(
                                d_model=d_model,
                nhead=n_heads,
                dim_feedforward=d_model * 4,
                                dropout=0.1,
                activation='gelu',
                batch_first=True,
            ) for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        # Weight Tying (GPT-2 핵심 기법)
        self.head.weight = self.tok_emb.weight

        def forward(self, idx):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
                x = self.tok_emb(idx) + self.pos_emb(pos)
        
        # Causal mask 생성
                causal_mask = nn.Transformer.generate_square_subsequent_mask(T, device=idx.device)
                for block in self.blocks:
                        x = block(x, memory=x, tgt_mask=causal_mask)
        
                x = self.ln_f(x)
                logits = self.head(x)
        return logits

    @torch.no_grad()
        def generate(self, idx, max_new_tokens=100, temperature=1.0, top_k=50):
        """자기회귀 텍스트 생성"""
                for _ in range(max_new_tokens):
            idx_cond = idx[:, -256:]  # 최근 256토큰만 사용
                        logits = self(idx_cond)[:, -1, :] / temperature
            
            # Top-K 샘플링 (GPT-2 기법)
            if top_k > 0:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float('-inf')
            
                        probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_token], dim=1)
        return idx


# ── 학습 루프 (Next-token Prediction) ──
def train_gpt(model, data_loader, epochs=5, lr=3e-4):
    """GPT 스타일 언어모델 학습"""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs * len(data_loader))
    
    model.train()
        for epoch in range(epochs):
                total_loss = 0
                for batch_idx, (x,) in enumerate(data_loader):
            # x: (batch, seq_len) — 입력 시퀀스
            # y: x를 한 칸 shift한 것이 정답
                        logits = model(x)  # (batch, seq, vocab)
            
            # Next-token prediction loss
                        loss = F.cross_entropy(
                logits[:, :-1, :].contiguous().view(-1, logits.size(-1)),
                x[:, 1:].contiguous().view(-1),
            )
            
            optimizer.zero_grad()
            loss.backward()
            # Gradient Clipping (GPT 학습 핵심)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            
            total_loss += loss.item()
            
            if batch_idx % 100 == 0:
                ppl = torch.exp(loss).item()
                                print(f"  Step {batch_idx}: loss={loss.item():.4f}, perplexity={ppl:.1f}")
        
                avg_loss = total_loss / len(data_loader)
        ppl = torch.exp(torch.tensor(avg_loss)).item()
                print(f"Epoch {epoch+1}: avg_loss={avg_loss:.4f}, perplexity={ppl:.1f}")

# 사용 예시
model = SimpleGPT(vocab_size=50257, d_model=256, n_heads=4, n_layers=4)
params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {params:,}")
x = torch.randint(0, 50257, (2, 64))
out = model(x)
print(f"Input: {x.shape} → Output: {out.shape}")